In [1]:
import subprocess
from datetime import datetime

In [2]:
def submit_slurm_job(command: str, name="dname"):
    """
    Submits a lightweight slurm job for a long time. 
    """
    now=datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    slurm_cmd = [
        "sbatch",
        "--partition=ycga",
        "--time=1:00:00",
        f"--output=master_{name}_{now}.out",
        "-c 1",
        f"-J {name}_master",
        "--wrap", command
    ]
    result = subprocess.run(slurm_cmd, capture_output=True, text=True)
    print(result.stdout.strip() if result.returncode == 0 else result.stderr.strip())

def bench(model_code):
    """
    Executes a particular model design & collects statistics. 
    """
    #spawns a master job to control the dask cluster 
    submit_slurm_job(f"""
    module load miniconda
    conda activate biopython
    python cluster.py {model_code}
    """,name=model_code)

def evaluate_model():
    pass

def evaluate_performance():
    pass

In [11]:
bench("c900090")

Submitted batch job 50617991


Below to be moved

In [14]:



#print(create_matricies(main_form=modelspecs.iloc[0]["Equation main"],zin_form=modelspecs.iloc[0]["Equation Z"],data=sh_dat))

def create_cluster():
    """
    Makes a simple slurm cluster with some preset parameters.
    """

    

    return cluster,client

def kill_cluster(client):
    client.shutdown()

#we will start up the cluster, run, kill for each, then get job statistics from sacct
#each cluster will get an ID & put it in the name of the job & save that ID for later 
#sacct summary.. 
#collect with subprocess query

@delayed
def statsmodels_fit(X,y,Z):
    zinb_model = smdc.ZeroInflatedNegativeBinomialP(y, X, exog_infl=Z)

    n_count_params = zinb_model.exog.shape[1]      # Count model parameters
    n_infl_params = zinb_model.exog_infl.shape[1]    # Inflation model parameters
    n_total = n_count_params + n_infl_params + 1 # adding 1 for alpha
    start_params = np.full(n_total, 0.1)

    zinb_result = zinb_model.fit(start_params=start_params,maxiter=1000,method="cg")

    return zinb_result

def bench(row):
    print(f"[+] Fitting {modelspecs.iloc[0].name}",flush=True)
    print(f"[+] Creating cluster")
    #if "Statsmodels" in modelspecs.iloc[row]['Hardware']:
    #    print("[+] No GPUs needed.")
    cluster,client=create_cluster()

    print("[+] Scattering data")
    sh_dat_fut = client.scatter(sh_dat, broadcast=True)

    print("[+] Creating matricies")
    mats = create_matricies(
        main_form=modelspecs.iloc[row]["Equation main"],
        zin_form=modelspecs.iloc[row]["Equation Z"],
        data=sh_dat_fut
    )

    X, y, Z=mats.compute()

    print("[+] Matricies done. Fitting model")

    model=None

    #little rats-nest to handle a couple possibilities...
    if pd.isnull(modelspecs.iloc[row]['Broken_by']):
        #model not parallelizably 

        if "Statsmodels" in modelspecs.iloc[row]['Hardware']:
            print("[+] Creating statsmodels model")
            model=statsmodels_fit(X,y,Z)

    print("[+] Fitting...")

    model=model.compute()

    print("[+] Done!")

    print(f"job_ids: {cluster.job_ids}")

    kill_cluster(client)
    

    return model

    


In [5]:
model=bench(row=0)

TypeError: bench() got an unexpected keyword argument 'row'

In [ ]:
print(model)
import pickle
import datetime

# 1. Print the current time
now = datetime.datetime.now()
print(f"Current time: {now.strftime('%Y-%m-%d %H:%M:%S')}")



# 3. Pickle and dump the object to disk
with open('c00000.pkl', 'wb') as f:
    pickle.dump(model, f)

print()

In [ ]:
#print("[+] Dumping model to disc.")

#print("[+] Collecting statistics...")

#print("[+] Dumping statistics")